In [1]:
%pylab inline
import sys, os
import matplotlib.pylab as pylab
pylab.rcParams['figure.figsize'] = 8, 6  # that's default image size for this interactive session
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
from ipywidgets.widgets import interactive, fixed, interact
%config InlineBackend.figure_format = 'retina'

Populating the interactive namespace from numpy and matplotlib


In [2]:
hr = genfromtxt('CsPBI3_hr.dat', skip_header=52)
print(hr.shape)

(104976, 7)


In [3]:
import pandas as pd 
df = pd.DataFrame(hr)
df = df.iloc[:, 0:3]
df.drop_duplicates()

,0,1,2
0,-4.0,-4.0,-4.0
144,-4.0,-4.0,-3.0
288,-4.0,-4.0,-2.0
432,-4.0,-4.0,-1.0
576,-4.0,-4.0,0.0
...,...,...,...
104256,4.0,4.0,0.0
104400,4.0,4.0,1.0
104544,4.0,4.0,2.0
104688,4.0,4.0,3.0


In [4]:
# This reads the hamiltonian elements for a specific unit cell coordinate.  
# Wannier90 reports them by unit cell (x,y,z, i,j) where (1,0,0, 1,1) is the overlap of
# wavefunction 1 in the 0,0,0 cell with wavefunction 1 translated by one unit cell in the x direction.
def GetOneCell(x,y,z):
    hrtrim = hr[(hr[:,0]==x) & (hr[:,1]==y) & (hr[:,2]==z)]
    hrtrim = hrtrim[:,3:]
    H = np.zeros((12,12), dtype='complex')
    for r in hrtrim[:]:
        i = int(r[0])-1
        j = int(r[1])-1
        overlap = complex(r[2], r[3])
        H[i, j] = overlap
    return H



In [5]:
# Print the eigenvalues for just the consideration of one unit cell.
H = GetOneCell(0,0,0)
print(eig(H)[0])

[5.64580188+0.j 5.47953312+0.j 5.44779236+0.j 2.67230971+0.j
 1.73244027+0.j 1.83736853+0.j 1.90108719+0.j 2.0275013 +0.j
 2.09679739+0.j 2.57730144+0.j 2.3468634 +0.j 2.34214441+0.j]


In [40]:
# Now do the same but considering the 6 nearest neighbors.
H = GetOneCell(0,0,0)

H += GetOneCell(1,0,0)
H += GetOneCell(0,1,0)
H += GetOneCell(0,0,1)
H += GetOneCell(-1,0,0)
H += GetOneCell(0,-1,0)
H += GetOneCell(0,0,-1)

print(eig(H)[0])

[6.92609211+0.j 6.58758006+0.j 6.22664337+0.j 1.27883179+0.j
 1.50693699+0.j 2.63325513+0.j 2.47609966+0.j 2.40683117+0.j
 2.21971184+0.j 2.09780902+0.j 1.84270513+0.j 1.88485073+0.j]


In [41]:
# Now build it using every element.
H = np.zeros((12,12), dtype='complex')
for r in hr[:]:
    x = int(r[0])
    y = int(r[1])
    z = int(r[2])
    i = int(r[3])-1
    j = int(r[4])-1
    overlap = complex(r[5], r[6])
    H[i, j] += overlap

print(eig(H)[0])

 

[ 6.72950221-5.34175679e-20j  7.85918347+6.82805960e-18j
  7.87370798+1.65977316e-17j -0.2837003 -1.31635345e-17j
 -0.29876321-1.99997699e-17j -0.29330765+2.35825679e-17j
  2.247901  -2.17431897e-18j  2.24098973+6.29342882e-19j
  2.24423419-9.07483142e-18j  2.09593496-3.91335038e-18j
  2.09358203-1.26323100e-19j  2.09014658-2.69647080e-18j]


In [43]:
pd.DataFrame(H)

,0,1,2,3,4,5,6,7,8,9,10,11
0,1.984117+0.000000j,0.654384+0.000302j,-0.084960-0.000004j,0.175677+0.000004j,-0.250318-0.000034j,0.377121+0.000143j,-0.023766+0.000000j,-0.168936-0.000088j,-0.061721+0.000002j,-0.133589+0.000001j,0.730852+0.000215j,0.088409+0.000000j
1,0.654384-0.000302j,5.517131+0.000000j,0.247061-0.000064j,0.415257-0.000211j,1.830705-0.000611j,-0.399378-0.000035j,0.847246-0.000209j,-0.856116+0.000184j,0.512417-0.000224j,1.979264-0.000419j,-0.200701+0.000009j,-0.742985+0.000319j
2,-0.084960+0.000004j,0.247061+0.000064j,2.033900+0.000000j,0.022833+0.000003j,-0.033175-0.000006j,-0.359717-0.000608j,-0.118920+0.000000j,0.059435+0.000034j,-0.250093-0.000036j,-0.019247-0.000006j,0.681667+0.000249j,0.017518+0.000000j
3,0.175677-0.000004j,0.415257+0.000211j,0.022833-0.000003j,1.995699+0.000000j,0.094706-0.000009j,-1.261079-0.000600j,-0.056620-0.000017j,0.082969-0.000032j,0.206176+0.000017j,-0.127368-0.000008j,-1.096548-0.000516j,-0.032274+0.000000j
4,-0.250318+0.000034j,1.830705+0.000611j,-0.033175+0.000006j,0.094706+0.000009j,1.873533+0.000000j,1.070780+0.001068j,-0.130552+0.000027j,-0.216664-0.000084j,-0.024107+0.000005j,-0.126641+0.000004j,-0.554004-0.000111j,0.239656-0.000023j
5,0.377121-0.000143j,-0.399378+0.000035j,-0.359717+0.000608j,-1.261079+0.000600j,1.070780-0.001068j,5.583275+0.000000j,-0.635325+0.000580j,2.677163-0.000894j,-0.524084+0.000461j,0.048074-0.000045j,-0.168652-0.000021j,-0.166943+0.000560j
6,-0.023766+0.000000j,0.847246+0.000209j,-0.118920+0.000000j,-0.056620+0.000017j,-0.130552-0.000027j,-0.635325-0.000580j,1.995459+0.000000j,0.258176+0.000131j,-0.035802-0.000018j,-0.152123-0.000013j,0.811441+0.000404j,0.055637+0.000002j
7,-0.168936+0.000088j,-0.856116-0.000184j,0.059435-0.000034j,0.082969+0.000032j,-0.216664+0.000084j,2.677163+0.000894j,0.258176-0.000131j,1.940010+0.000000j,0.066951-0.000016j,0.121167-0.000063j,0.212450-0.000081j,0.052317+0.000008j
8,-0.061721-0.000002j,0.512417+0.000224j,-0.250093+0.000036j,0.206176-0.000017j,-0.024107-0.000005j,-0.524084-0.000461j,-0.035802+0.000018j,0.066951+0.000016j,1.963236+0.000000j,0.087251-0.000007j,2.798225+0.000271j,0.059154-0.000001j
9,-0.133589-0.000001j,1.979264+0.000419j,-0.019247+0.000006j,-0.127368+0.000008j,-0.126641-0.000004j,0.048074+0.000045j,-0.152123+0.000013j,0.121167+0.000063j,0.087251+0.000007j,1.922518+0.000000j,-0.953583-0.000292j,0.192119-0.000008j


In [49]:
eig(H)[1][1]


array([ 0.63099304+0.00000000e+00j, -0.37324883-4.93372689e-03j,
       -0.4777017 -8.48345080e-03j, -0.28742367-3.25222056e-03j,
       -0.02491252-1.39129398e-02j, -0.38152867-2.78025486e-03j,
        0.02595097-1.23587070e-05j,  0.02151139-6.41223864e-04j,
       -0.06295837+2.07350459e-04j,  0.00705443-1.86670317e-05j,
        0.00113857-3.81341499e-05j,  0.00879809+1.11072419e-05j])

In [44]:
# Print out the energy levels from greatest to least, 
# along with <psi_m|psi_n> so we can see what orbitals
# they represent.
E = linalg.eig(H)[0]
psi = linalg.eig(H)[1]

for i in range(len(E)):
    print(f'Energy = {np.abs(E[i]):0.2f} eV')
    print(f'psi=(', end='')
    normval=np.abs(psi[i]).sum()
    for j in range(len(E)):
        print(f'{np.abs(psi[i][j])/normval:0.2f}, ', end='')
    print(')')

Energy = 6.73 eV
psi=(0.05, 0.02, 0.02, 0.10, 0.08, 0.03, 0.14, 0.15, 0.11, 0.07, 0.24, 0.01, )
Energy = 7.86 eV
psi=(0.27, 0.16, 0.21, 0.12, 0.01, 0.17, 0.01, 0.01, 0.03, 0.00, 0.00, 0.00, )
Energy = 7.87 eV
psi=(0.01, 0.04, 0.02, 0.11, 0.03, 0.03, 0.18, 0.09, 0.17, 0.29, 0.04, 0.01, )
Energy = 0.28 eV
psi=(0.04, 0.03, 0.06, 0.05, 0.12, 0.08, 0.16, 0.09, 0.08, 0.18, 0.10, 0.02, )
Energy = 0.30 eV
psi=(0.10, 0.01, 0.06, 0.06, 0.11, 0.15, 0.10, 0.13, 0.04, 0.00, 0.10, 0.13, )
Energy = 0.29 eV
psi=(0.25, 0.32, 0.09, 0.03, 0.23, 0.01, 0.01, 0.04, 0.02, 0.00, 0.00, 0.01, )
Energy = 2.25 eV
psi=(0.03, 0.06, 0.01, 0.13, 0.06, 0.04, 0.15, 0.19, 0.03, 0.03, 0.01, 0.25, )
Energy = 2.24 eV
psi=(0.06, 0.11, 0.06, 0.06, 0.22, 0.05, 0.11, 0.10, 0.06, 0.11, 0.05, 0.01, )
Energy = 2.24 eV
psi=(0.06, 0.10, 0.09, 0.22, 0.01, 0.11, 0.02, 0.04, 0.16, 0.01, 0.07, 0.10, )
Energy = 2.10 eV
psi=(0.07, 0.01, 0.09, 0.05, 0.01, 0.20, 0.03, 0.16, 0.11, 0.01, 0.11, 0.15, )
Energy = 2.09 eV
psi=(0.12, 0.18, 0.31, 

In [2]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interactive

# Function to plot the hyperbola based on the eccentricity
def plot_hyperbola(e):
    plt.figure(figsize=(10, 6))
    
    # Hyperbola parameters
    a = 1  # semi-major axis
    b = np.sqrt(e**2 - 1)  # semi-minor axis based on eccentricity
    
    # Generate x values
    x = np.linspace(-3 * a, 3 * a, 400)
    
    # Compute y values for the hyperbola branches
    y1 = b * np.sqrt((x**2 / a**2) - 1)
    y2 = -y1
    
    # Plotting the hyperbola
    plt.plot(x, y1, label=f'Branch 1 (e={e:.2f})', color='blue')
    plt.plot(x, y2, label='Branch 2', color='blue')
    
    # Plot settings
    plt.xlim(-3 * a, 3 * a)
    plt.ylim(-3 * b, 3 * b)
    plt.axhline(0, color='black', lw=0.5, ls='--')
    plt.axvline(0, color='black', lw=0.5, ls='--')
    plt.title('Hyperbola with Eccentricity = {:.2f}'.format(e))
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid()
    plt.legend()
    plt.show()

# Slider for eccentricity
eccentricity_slider = FloatSlider(value=2.0, min=1.0, max=5.0, step=0.1, description='Eccentricity (e):')

# Create interactive plot
interactive_plot = interactive(plot_hyperbola, e=eccentricity_slider)
output = interactive_plot.children[-1]
output.layout.height = '400px'
interactive_plot


interactive(children=(FloatSlider(value=2.0, description='Eccentricity (e):', max=5.0, min=1.0), Output(layout…

In [6]:
import ipywidgets as widgets
from ipywidgets import HBox, VBox
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
%matplotlib widget

In [7]:
@widgets.interact
def f(x=5):
    print(x)

interactive(children=(IntSlider(value=5, description='x', max=15, min=-5), Output()), _dom_classes=('widget-in…